# Mini-Project 4 — A LangGraph Alert-Triage Agent
### IT7075: Applied AI for Cybersecurity · LangGraph

A chain runs straight through. An **agent decides** — and the decision is the interesting part,
because a triage agent can be wrong in two very different ways:

- it **misses** a real incident (the dangerous error), or
- it **escalates noise** and buries your analysts (the expensive error).

You will build the Lecture 3 triage graph, run it against **12 labelled alerts**, and measure both
kinds of error. Then you tune the rule and watch one error trade against the other.

**This project is mostly analysis, not coding.** There are five TODOs and they total **ten lines** —
the graph is small on purpose. Most of the credit is for what you write about the results.

| | LangGraph concept | You write |
|---|---|---|
| **TODO 1** | the **state** (a `TypedDict`) | 2 lines |
| **TODO 2** | a **node** (plain function → dict) | 2 lines |
| **TODO 3** | the **router** for a conditional edge | 1 line |
| **TODO 4** | a node that calls a **tool** | 2 lines |
| **TODO 5** | **build and compile** the graph | 3 lines |

Find them by searching for `###################`:

```
###################  TODO n — title  ###################
# what to do, and how
################
```

> **Runs fully offline.** Every node is ordinary Python — no API key, no model download, no GPU.
> That is deliberate: the point is the *routing decision*, and a deterministic agent lets you
> measure it exactly.

Student implementation by Colin Vitols, with AI assistance. Parts 1-8 complete the course practice in this copy. Parts 9 onward contain the custom experiment. The CVE identifier in the practice lookup is an instructor-provided fictional stub, not verified vulnerability intelligence. All project decisions are deterministic Python; no LLM is used.

## Part 1 — The state  ·  **TODO 1**

The state is the graph's memory: one dictionary that every node reads from and writes to. Declaring
it as a `TypedDict` is how you say *these* are the fields an alert carries as it moves through the
graph.

This is the first real difference from the LangChain module. There, data flowed along a pipe from
one step to the next. Here every node sees the **whole** state, and a node only returns the keys it
changed.

> **Expected output:** `state fields: ['alert', 'risk', 'cve_info', 'report']`.

In [1]:
import json, os
from pathlib import Path
from typing import TypedDict

PROJECT = Path.cwd() if Path("my_alerts.json").exists() else Path.cwd() / "langgraph_project"
assert (PROJECT / "my_alerts.json").is_file(), "Run from the repository root or project folder"
PROJECT = PROJECT.resolve()
ALERTS_FILE = PROJECT.parent / "06_LangGraph/assignments/project/alerts.json"


###################  TODO 1 — declare the state  ###################
# WHAT: add the two missing fields to TriageState.
#
# The agent needs four things in its state, and two are already there:
#     alert     : str   the incoming alert text            (given)
#     risk      : str   "high" or "low", set by TODO 2     <- you add this
#     cve_info  : str   what the tool found, set by TODO 4 <- you add this
#     report    : str   the final summary                  (given)
#
# HOW (the Lecture 2 "Define the State" cell): inside the class, one line per
# field, in the form   name: type
#
# WHY a TypedDict and not a plain dict: it documents the agent's memory in one
# place, so when a node returns {"risk": "high"} you can see where that lands.
################

class TriageState(TypedDict):
    alert: str
    risk: str
    cve_info: str
    report: str


print("state fields:", list(TriageState.__annotations__))


state fields: ['alert', 'risk', 'cve_info', 'report']


## Part 2 — The classify node  ·  **TODO 2**

A node is just a Python function: it takes the state and returns a dict of the keys it changed.

This particular node is the one your whole experiment turns on. It decides `risk` by looking for
words in the alert text — the same rule as the Lecture 3 notebook. Keep `KEYWORDS` as a variable:
Part 7 changes it and re-measures.

> **Expected output:** `high` for the RCE alert, `low` for the failed-login alert.

In [2]:
# The Lecture 3 keyword list. Part 7 will change this and re-measure.
KEYWORDS = ["rce", "ransomware", "exfil", "critical", "admin"]


###################  TODO 2 — write the classify node  ###################
# WHAT: finish classify() so it sets `risk` from the alert text.
#
# HOW (the Lecture 3 "Conditional Edges" cell):
#   * is any keyword present?  Search the LOWERCASED alert text:
#         any(word in state["alert"].lower() for word in KEYWORDS)
#   * a node returns a DICT of just the fields it changed -- here:
#         {"risk": "high"}  or  {"risk": "low"}
#     Do NOT return the whole state, and do not mutate it in place.
#
# NOTE: matching is substring-based, so "admin" also fires on "administrator".
#       Keep that in mind when you read your results -- it matters later.
################

def classify(state: TriageState) -> dict:
    hit = any(word in state["alert"].lower() for word in KEYWORDS)                                                         # <<< 1 line
    return {"risk": "high" if hit else "low"}                                                        # <<< 1 line


# ---- given: try it on two alerts ----------------------------------------
for text in ["Critical RCE exploit attempt against the Apache server",
             "Single failed login from a known corporate device"]:
    print("%-58s -> %s" % (text[:56], classify({"alert": text})["risk"]))


Critical RCE exploit attempt against the Apache server     -> high
Single failed login from a known corporate device          -> low


## Part 3 — The router  ·  **TODO 3**

This is what makes it a graph rather than a chain. A **router** is a function that looks at the
state and returns a *label* — the name of the branch to take next. LangGraph calls it after the
`classify` node and jumps to whichever node that label points at.

The router decides nothing else: no work, no side effects, just "which way".

> **Expected output:** `high -> investigate`, `low -> quick_close`.

In [3]:
###################  TODO 3 — write the router  ###################
# WHAT: return the LABEL of the next branch, based on state["risk"].
#
#     risk == "high"  ->  "investigate"
#     anything else   ->  "quick_close"
#
# HOW (the Lecture 3 route_by_risk function): one line, e.g.
#     return "investigate" if <condition> else "quick_close"
#
# These two strings are labels, not node names -- TODO 5 maps them to real
# nodes. Keeping them separate is what lets you rewire a branch without
# touching this function.
################

def route_by_risk(state: TriageState) -> str:
    return "investigate" if state["risk"] == "high" else "quick_close"                                                         # <<< 1 line


# ---- given ---------------------------------------------------------------
for r in ["high", "low"]:
    print("%-5s -> %s" % (r, route_by_risk({"risk": r})))


high  -> investigate
low   -> quick_close


## Part 4 — A node that calls a tool  ·  **TODO 4**

A **tool** is just a function the agent can call for information it does not have. `lookup_cve` is
given — a tiny stand-in for a threat-intel API. Your job is the node that uses it and writes the
result into the state.

> **Expected output:** the RCE alert returns the `CVE-2026-0001` line; an unmatched alert returns
> the "No specific CVE found" line.

In [4]:
def lookup_cve(keyword: str) -> str:
    """GIVEN: the tool. A stand-in for a threat-intel lookup."""
    db = {"rce": "CVE-2026-0001: critical RCE; patch immediately.",
          "ransomware": "Multiple CVEs; isolate the host and restore from backup.",
          "credential": "Credential theft; force password resets and review logins.",
          "outbound": "Possible data theft; check egress logs and block the destination."}
    for key, value in db.items():
        if key in keyword.lower():
            return value
    return "No specific CVE found; follow standard triage."


###################  TODO 4 — write the investigate node  ###################
# WHAT: call the tool, then return BOTH fields it fills in.
#
# HOW:
#   * info = lookup_cve(state["alert"])
#   * return a dict with two keys:
#         "cve_info" -> info
#         "report"   -> "ESCALATE. " + info
#
# This is the whole idea of a tool: the node does not guess what the CVE is,
# it asks something that knows. Note the node returns two keys at once --
# a node may update as many state fields as it likes.
################

def investigate(state: TriageState) -> dict:
    info = lookup_cve(state["alert"])                                                        # <<< 1 line
    return {"cve_info": info, "report": "ESCALATE. " + info}                                                        # <<< 1 line


def quick_close(state: TriageState) -> dict:
    """GIVEN: the other branch. No tool call, no escalation."""
    return {"cve_info": "", "report": "CLOSE. Logged for monitoring; no action needed."}


# ---- given ---------------------------------------------------------------
for text in ["Critical RCE exploit attempt", "Backup job completed with a warning"]:
    print("%-32s -> %s" % (text, investigate({"alert": text})["cve_info"]))


Critical RCE exploit attempt     -> CVE-2026-0001: critical RCE; patch immediately.
Backup job completed with a warning -> No specific CVE found; follow standard triage.


## Part 5 — Build and compile the graph  ·  **TODO 5**

Now wire the pieces together. The nodes are added for you; you supply the three lines that make it
a runnable graph — where it starts, how the branch works, and the compile step.

```
        classify
           |  route_by_risk
     +-----+-----+
     |           |
investigate   quick_close
     |           |
     +-----+-----+
           END
```

> **Expected output:** the RCE alert reports `ESCALATE. CVE-2026-0001 ...`; the failed-login alert
> reports `CLOSE. Logged for monitoring ...`.

In [5]:
from langgraph.graph import StateGraph, END


def build_triage_graph():
    """Assemble the graph. Called again in Part 7 after KEYWORDS changes."""
    builder = StateGraph(TriageState)
    builder.add_node("classify", classify)
    builder.add_node("investigate", investigate)
    builder.add_node("quick_close", quick_close)

    ###################  TODO 5 — wire it up  ###################
    # WHAT: three lines, all from the Lecture 3 "Wire the Branching Graph" cell.
    #
    #   1. entry point -- the graph starts at the "classify" node:
    #          builder.set_entry_point("classify")
    #
    #   2. the conditional edge -- after "classify", call your router and map each
    #      LABEL it can return to the node that should run next:
    #          builder.add_conditional_edges("classify", route_by_risk,
    #                                        {"investigate": "investigate",
    #                                         "quick_close": "quick_close"})
    #      (left side = the label from TODO 3, right side = the node name)
    #
    #   3. compile it into a runnable graph and return it:
    #          return builder.compile()
    #
    # The two edges to END are already there: both branches finish the run.
    ################

    builder.set_entry_point("classify")                                                               # <<< 1 line
    builder.add_conditional_edges("classify", route_by_risk, {"investigate": "investigate", "quick_close": "quick_close"})                                                               # <<< 1 line

    builder.add_edge("investigate", END)
    builder.add_edge("quick_close", END)
    return builder.compile()                                                               # <<< 1 line


graph = build_triage_graph()


# ---- given: a blank state, then one run per alert ------------------------
def run(graph, alert_text):
    """GIVEN: invoke the graph on one alert and hand back the finished state."""
    return graph.invoke({"alert": alert_text, "risk": "", "cve_info": "", "report": ""})


for text in ["Critical RCE exploit attempt against the Apache server",
             "Single failed login from a known corporate device"]:
    out = run(graph, text)
    print("ALERT :", text)
    print("  risk:", out["risk"], "| report:", out["report"], "\n")


ALERT : Critical RCE exploit attempt against the Apache server
  risk: high | report: ESCALATE. CVE-2026-0001: critical RCE; patch immediately. 

ALERT : Single failed login from a known corporate device
  risk: low | report: CLOSE. Logged for monitoring; no action needed. 



## Part 6 — Score the agent  *(no TODO — run it and read it)*

The coding is done. Everything from here is measurement and interpretation.

`alerts.json` holds **12 real-looking alerts**, each labelled with what a SOC analyst would
actually do: **6 `escalate`** and **6 `close`**. Comparing your agent's route to that label gives
three numbers — and the two errors are not equally bad:

| | Meaning | Cost |
|---|---|---|
| **caught** | escalated something that deserved it | what you want |
| **missed** | closed a real incident | an attacker keeps going, unnoticed |
| **over-escalated** | escalated routine noise | analyst time, and eventually alert fatigue |

> **Expected output with the Lecture 3 keyword list: caught 3/6, missed 3/6, over-escalated 2/6.**
> The agent you just built misses **half** of the real incidents. Part 7 asks what to do about it.

In [6]:
ALERTS = json.load(open(ALERTS_FILE))["alerts"]
print("loaded", len(ALERTS), "labelled alerts\n")


def score(graph, alerts):
    """GIVEN: run every alert and compare the route with the analyst's label."""
    caught = missed = over = 0
    rows = []
    for a in alerts:
        out = run(graph, a["alert"])
        route = "escalate" if out["risk"] == "high" else "close"
        if a["expected"] == "escalate":
            verdict = "caught" if route == "escalate" else "MISSED"
        else:
            verdict = "over-escalated" if route == "escalate" else "closed ok"
        caught += verdict == "caught"
        missed += verdict == "MISSED"
        over += verdict == "over-escalated"
        rows.append({"id": a["id"], "expected": a["expected"], "agent": route,
                     "verdict": verdict, "alert": a["alert"][:52]})
    return caught, missed, over, rows


caught, missed, over, rows = score(graph, ALERTS)
n_esc = sum(a["expected"] == "escalate" for a in ALERTS)
n_close = len(ALERTS) - n_esc

print("caught          %d/%d real incidents" % (caught, n_esc))
print("MISSED          %d/%d real incidents      <-- the dangerous error" % (missed, n_esc))
print("over-escalated  %d/%d routine alerts      <-- the expensive error\n" % (over, n_close))

print("%-5s %-9s %-9s %-15s %s" % ("id", "expected", "agent", "verdict", "alert"))
for r in rows:
    print("%-5s %-9s %-9s %-15s %s" % (r["id"], r["expected"], r["agent"], r["verdict"], r["alert"]))

loaded 12 labelled alerts

caught          3/6 real incidents
MISSED          3/6 real incidents      <-- the dangerous error
over-escalated  2/6 routine alerts      <-- the expensive error

id    expected  agent     verdict         alert
A01   escalate  escalate  caught          Critical RCE exploit attempt against the internet-fa
A02   escalate  escalate  caught          Ransomware canary file modified on the finance file 
A03   escalate  close     MISSED          Large outbound data transfer to an unknown host at 0
A04   escalate  escalate  caught          New local administrator account created on the domai
A05   escalate  close     MISSED          Credential dumping tool signature detected in memory
A06   escalate  close     MISSED          Repeated MFA prompts followed by a successful login 
A07   close     escalate  over-escalated  Scheduled vulnerability scan reported critical findi
A08   close     close     closed ok       User reset their own password via the self-service p
A

## Part 7 — Tune the rule and watch the two errors trade  *(no TODO)*

The obvious fix for a missed incident is to add words. Do that four times over and watch what it
costs.

> **Expected output:**
>
> | keyword list | caught | missed | over-escalated |
> |---|---:|---:|---:|
> | narrow (2 words) | 2/6 | **4/6** | 0/6 |
> | **lecture (5 words)** | 3/6 | **3/6** | 2/6 |
> | wider (8 words) | **6/6** | 0/6 | 2/6 |
> | widest (11 words) | 6/6 | 0/6 | **4/6** |
>
> Read the two error columns against each other. Going from narrow to widest drives missed
> detections to zero and doubles the false escalations. There is no row with zeroes in both.
>
> **And look at *which* alerts stay wrong.** A07 and A12 are over-escalated by every list that
> catches all six real incidents. A07 says *critical* — about findings from our own scanner on a
> sandbox VM. A12 says *Administrator* — about an approved patch in a maintenance window. The
> trigger word is right there in both; what makes them harmless is **context the words do not
> carry**. No keyword list can fix that, which is the most important thing this table has to say.

In [7]:
KEYWORD_LISTS = {
    "narrow (2 words)":   ["rce", "ransomware"],
    "lecture (5 words)":  ["rce", "ransomware", "exfil", "critical", "admin"],
    "wider (8 words)":    ["rce", "ransomware", "exfil", "critical", "admin",
                           "outbound", "credential", "mfa"],
    "widest (11 words)":  ["rce", "ransomware", "exfil", "critical", "admin",
                           "outbound", "credential", "mfa", "failed", "scan", "password"],
}

print("%-20s %-8s %-8s %-15s %s" % ("keyword list", "caught", "missed", "over-escalated", "which ones"))
sweep = []
for name, words in KEYWORD_LISTS.items():
    KEYWORDS[:] = words                      # the classify node reads this list
    c, m, o, rr = score(build_triage_graph(), ALERTS)
    bad = [r["id"] for r in rr if r["verdict"] == "MISSED"]
    fat = [r["id"] for r in rr if r["verdict"] == "over-escalated"]
    sweep.append({"list": name, "caught": c, "missed": m, "over": o})
    print("%-20s %-8s %-8s %-15s missed=%-14s over=%s"
          % (name, "%d/%d" % (c, n_esc), "%d/%d" % (m, n_esc), "%d/%d" % (o, n_close),
             ",".join(bad) or "none", ",".join(fat) or "none"))

KEYWORDS[:] = KEYWORD_LISTS["lecture (5 words)"]     # put it back
print("\nSaved to results_practice.csv")
with open(str(PROJECT / "results_practice.csv"), "w", encoding="utf-8") as f:
    f.write("keyword_list,caught,missed,over_escalated\n")
    for s in sweep:
        f.write("%s,%d,%d,%d\n" % (s["list"], s["caught"], s["missed"], s["over"]))


keyword list         caught   missed   over-escalated  which ones
narrow (2 words)     2/6      4/6      0/6             missed=A03,A04,A05,A06 over=none
lecture (5 words)    3/6      3/6      2/6             missed=A03,A05,A06    over=A07,A12
wider (8 words)      6/6      0/6      2/6             missed=none           over=A07,A12
widest (11 words)    6/6      0/6      4/6             missed=none           over=A07,A08,A10,A12

Saved to results_practice.csv


## Part 8 — Memory: why the graph beats a chain  *(no TODO)*

One more LangGraph feature from Lecture 3. Compile the **same** graph with a `MemorySaver` and every
run tagged with a `thread_id` is remembered, so an incident can be picked up later — something a
one-shot chain cannot do.

> **Expected output:** the state saved under `incident-42` is still readable after the run, showing
> the alert, the risk, and the report the agent produced.

In [8]:
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(TriageState)
builder.add_node("classify", classify)
builder.add_node("investigate", investigate)
builder.add_node("quick_close", quick_close)
builder.set_entry_point("classify")
builder.add_conditional_edges("classify", route_by_risk,
                              {"investigate": "investigate", "quick_close": "quick_close"})
builder.add_edge("investigate", END)
builder.add_edge("quick_close", END)
graph_with_memory = builder.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "incident-42"}}
out = graph_with_memory.invoke(
    {"alert": "Ransomware canary file modified on the finance file share",
     "risk": "", "cve_info": "", "report": ""}, config)
print("run     :", out["report"])

# ...later, or in another cell, the incident is still there:
saved = graph_with_memory.get_state(config).values
print("recalled:", saved["alert"])
print("          risk =", saved["risk"], "| cve_info =", saved["cve_info"][:48])

print("\nA different thread_id starts a fresh incident:")
other = {"configurable": {"thread_id": "incident-99"}}
print("  incident-99 has state yet?", bool(graph_with_memory.get_state(other).values))

run     : ESCALATE. Multiple CVEs; isolate the host and restore from backup.
recalled: Ransomware canary file modified on the finance file share
          risk = high | cve_info = Multiple CVEs; isolate the host and restore from

A different thread_id starts a fresh incident:
  incident-99 has state yet? False


## Part 9 - My twelve labelled alerts
The scenarios continue my earlier incident-triage projects, moving from retrieval to routing. Six need investigation and six are routine. `expected` and `note` are only for scoring: the graph receives text and structured event fields, never those answers. A06 has none of the tested keywords; A07 is routine but includes `admin`. Sources and individual rationales are in the accompanying files.

In [9]:
from collections import Counter
import csv
from copy import deepcopy
from datetime import datetime
from importlib.metadata import version

MINE = json.loads((PROJECT / "my_alerts.json").read_text())["alerts"]
CONTEXT = json.loads((PROJECT / "context_fixture.json").read_text())
assert len(MINE) == 12 and len({a["id"] for a in MINE}) == 12
assert Counter(a["expected"] for a in MINE) == {"escalate": 6, "close": 6}
assert all(a["note"] and a["alert"] for a in MINE)
print("Label counts:", dict(Counter(a["expected"] for a in MINE)))
for a in MINE:
    print(a["id"], a["expected"], "|", a["alert"], "\n  Why:", a["note"])


Label counts: {'escalate': 6, 'close': 6}
A01 escalate | IDS flagged an RCE payload against WEB-01 and EDR recorded the web process spawning a shell. 
  Why: Correlated network and process evidence indicates execution on a production web server.
A02 escalate | EDR detected ransomware behavior on FILE-02 as hundreds of documents were renamed and became unreadable. 
  Why: Active mass file modification threatens availability; this is not a scheduled test.
A03 escalate | Windows 4728 recorded an unapproved service account addition to Domain Admins on DC-01. 
  Why: An unexpected privileged group membership requires investigation; no matching approval exists.
A04 escalate | EDR detected credential dumping through an unsigned process reading LSASS memory on HR-04. 
  Why: A suspicious process reading authentication material can enable account compromise.
A05 escalate | Proxy logs show an outbound upload of payroll archives to a newly seen destination with no business owner. 
  Why: Sensitiv

## Part 10 - A baseline and a graph change
The baseline uses the lecture's conditional edge. The changed graph inserts `context_review` between classification and routing. Its ordinary Python tool looks up asset role and verifies a change against actor, asset, action, and an aware timestamp. Unapproved production audit shutdowns become high risk; matched group changes and lab test signatures become low risk. Other events keep the keyword decision. A backup lookup is deliberately absent, leaving a visible limitation.

This is one graph-level intervention with two context policies; the comparison measures their combined effect, not each policy's independent contribution. Both variants receive the same input fields and keywords. The graph never reads the ground-truth labels.

In [10]:
class AlertState(TypedDict):
    alert: str
    event: dict
    risk: str
    matched: list[str]
    context: dict
    reason: str
    decision: str
    report: str

BASE_WORDS = ("rce", "ransomware", "exfil", "critical", "admin")
OPERATING_WORDS = BASE_WORDS + ("credential", "outbound")


def lookup_context(event: dict) -> dict:
    """Offline stand-in for read-only inventory and change-management lookups."""
    asset = CONTEXT["assets"].get(event.get("asset"), {})
    change = CONTEXT["changes"].get(event.get("change_id"), {})
    approved = False
    if change.get("approved") is True:
        identity_matches = all(event.get(k) == change.get(k) for k in ("asset", "actor", "action"))
        try:
            moment, start, end = (datetime.fromisoformat(v) for v in
                                  (event["occurred_at"], change["start"], change["end"]))
            aware = all(t.tzinfo is not None and t.utcoffset() is not None for t in (moment, start, end))
            approved = identity_matches and aware and start <= moment <= end
        except (KeyError, TypeError, ValueError):
            approved = False
    return {"environment": asset.get("environment", "unknown"),
            "role": asset.get("role", "unknown"), "verified_change": approved}


def context_review(state: AlertState) -> dict:
    context = lookup_context(state["event"])
    action = state["event"].get("action")
    risk, reason = state["risk"], state["reason"]
    if (action == "StopLogging" and context["environment"] == "production"
            and context["role"] == "audit_trail" and not context["verified_change"]):
        risk, reason = "high", "Unapproved logging shutdown on a production audit asset."
    elif (context["verified_change"] and
          (action == "group_add" or (action == "test_signature" and context["environment"] == "lab"))):
        risk, reason = "low", "Actor, asset, action and time match a simulated approved change."
    return {"context": context, "risk": risk, "reason": reason}


def investigate_custom(state: AlertState) -> dict:
    return {"decision": "escalate", "report": "ESCALATE. " + state["reason"]}


def close_custom(state: AlertState) -> dict:
    return {"decision": "close", "report": "CLOSE. " + state["reason"]}


def build_custom(words, with_context=False, checkpointer=None):
    # Capture an immutable tuple: later experiments cannot silently retune this graph.
    words = tuple(words)
    def classify_custom(state: AlertState) -> dict:
        matched = [word for word in words if word in state["alert"].lower()]
        return {"matched": matched, "risk": "high" if matched else "low",
                "reason": "Keyword matches: " + ", ".join(matched) if matched else "No keyword matched.",
                "context": {}, "decision": "", "report": ""}
    builder = StateGraph(AlertState)
    builder.add_node("classify", classify_custom)
    builder.add_node("investigate", investigate_custom)
    builder.add_node("quick_close", close_custom)
    builder.set_entry_point("classify")
    route_from = "classify"
    if with_context:
        builder.add_node("context_review", context_review)
        builder.add_edge("classify", "context_review")
        route_from = "context_review"
    builder.add_conditional_edges(route_from, route_by_risk,
                                  {"investigate": "investigate", "quick_close": "quick_close"})
    builder.add_edge("investigate", END)
    builder.add_edge("quick_close", END)
    return builder.compile(checkpointer=checkpointer)


def graph_input(alert):
    return {"alert": alert["alert"], "event": deepcopy(alert["event"]), "risk": "",
            "matched": [], "context": {}, "reason": "", "decision": "", "report": ""}

baseline = build_custom(BASE_WORDS)
enhanced = build_custom(OPERATING_WORDS, with_context=True)
print(enhanced.get_graph().draw_mermaid())
(PROJECT / "graph.mmd").write_text(enhanced.get_graph().draw_mermaid() + "\n")


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	investigate(investigate)
	quick_close(quick_close)
	context_review(context_review)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify --> context_review;
	context_review -.-> investigate;
	context_review -.-> quick_close;
	investigate --> __end__;
	quick_close --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



516

## Part 11 - Measure one change at a time
K0 is the lecture rule. K1-K4 each add exactly one word to the preceding keyword configuration. G1 compares directly with K2: identical words, plus the context-review node/tool. Counts are caught incidents, missed incidents, and over-escalated routine alerts, each out of six. There is no combined accuracy score.

In [11]:
EXPERIMENTS = [
    ("K0", BASE_WORDS, False, "none", "Lecture baseline"),
    ("K1", BASE_WORDS + ("credential",), False, "K0", "Add credential"),
    ("K2", OPERATING_WORDS, False, "K1", "Add outbound"),
    ("K3", OPERATING_WORDS + ("failed",), False, "K2", "Add failed"),
    ("K4", OPERATING_WORDS + ("failed", "password"), False, "K3", "Add password"),
    ("G1", OPERATING_WORDS, True, "K2", "Add context-review node and lookup tool"),
]
all_words = set(word for _, words, *_ in EXPERIMENTS for word in words)
assert not any(word in MINE[5]["alert"].lower() for word in all_words)
assert any(word in MINE[6]["alert"].lower() for word in BASE_WORDS)


def evaluate(g, alerts):
    details = []
    for a in alerts:
        out = g.invoke(graph_input(a))
        outcome = ("caught" if out["decision"] == "escalate" else "missed") if a["expected"] == "escalate" else ("over_escalated" if out["decision"] == "escalate" else "closed_ok")
        details.append({"id": a["id"], "expected": a["expected"], "decision": out["decision"],
                        "outcome": outcome, "matched": out["matched"], "reason": out["reason"]})
    return details

results, evidence = [], {}
for name, words, use_context, compared_to, change in EXPERIMENTS:
    details = evaluate(build_custom(words, use_context), MINE)
    evidence[name] = details
    counts = Counter(r["outcome"] for r in details)
    row = {"configuration": name, "compared_to": compared_to, "change": change,
           "keywords": "|".join(words), "context_review": use_context,
           "caught": counts["caught"], "missed": counts["missed"],
           "over_escalated": counts["over_escalated"], "closed_ok": counts["closed_ok"],
           "missed_ids": "|".join(r["id"] for r in details if r["outcome"] == "missed"),
           "over_escalated_ids": "|".join(r["id"] for r in details if r["outcome"] == "over_escalated")}
    assert row["caught"] + row["missed"] == row["over_escalated"] + row["closed_ok"] == 6
    results.append(row)
    print(name, "caught", row["caught"], "/6 | missed", row["missed"], "/6 | over", row["over_escalated"], "/6",
          "| missed:", row["missed_ids"] or "none", "| over:", row["over_escalated_ids"] or "none")
with (PROJECT / "results_mine.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0])); writer.writeheader(); writer.writerows(results)
(PROJECT / "evaluation_details.json").write_text(json.dumps(evidence, indent=2) + "\n")
keyword_errors = [{r["id"] for r in evidence[name] if r["outcome"] in ("missed", "over_escalated")}
                  for name, *_ in EXPERIMENTS if name.startswith("K")]
print("Wrong at every tested keyword setting:", sorted(set.intersection(*keyword_errors)))
print("Graph-only change K2 -> G1:",
      {k: results[-1][k] - results[2][k] for k in ("caught", "missed", "over_escalated")})


K0 caught 3 /6 | missed 3 /6 | over 2 /6 | missed: A04|A05|A06 | over: A07|A08
K1 caught 4 /6 | missed 2 /6 | over 2 /6 | missed: A05|A06 | over: A07|A08
K2 caught 5 /6 | missed 1 /6 | over 3 /6 | missed: A06 | over: A07|A08|A09
K3 caught 5 /6 | missed 1 /6 | over 4 /6 | missed: A06 | over: A07|A08|A09|A11
K4 caught 5 /6 | missed 1 /6 | over 5 /6 | missed: A06 | over: A07|A08|A09|A10|A11
G1 caught 6 /6 | missed 0 /6 | over 1 /6 | missed: none | over: A09
Wrong at every tested keyword setting: ['A06', 'A07', 'A08']
Graph-only change K2 -> G1: {'caught': 1, 'missed': -1, 'over_escalated': -2}


## Part 12 - Follow the actual node updates
A01 shows a caught incident. A06 shows a baseline miss and the changed graph's recovery, even though no tested keyword appears. A07 shows a keyword false alarm that the context tool closes. A09 stays over-escalated in G1 because backup context is not integrated. Streaming records nodes that actually ran; the path is not inferred from the final label.

In [12]:
traces = {}
for name, g, alert in [("baseline_A01", baseline, MINE[0]), ("baseline_A06", baseline, MINE[5]),
                       ("G1_A06", enhanced, MINE[5]), ("G1_A07", enhanced, MINE[6]),
                       ("G1_A09", enhanced, MINE[8])]:
    events = list(g.stream(graph_input(alert), stream_mode="updates"))
    traces[name] = events
    print("\n", name, alert["alert"])
    for event in events:
        print(json.dumps(event, sort_keys=True))
(PROJECT / "traces.json").write_text(json.dumps(traces, indent=2) + "\n")



 baseline_A01 IDS flagged an RCE payload against WEB-01 and EDR recorded the web process spawning a shell.
{"classify": {"context": {}, "decision": "", "matched": ["rce"], "reason": "Keyword matches: rce", "report": "", "risk": "high"}}
{"investigate": {"decision": "escalate", "report": "ESCALATE. Keyword matches: rce"}}

 baseline_A06 CloudTrail recorded StopLogging for the production audit trail, invoked by an unfamiliar role with no change ticket.
{"classify": {"context": {}, "decision": "", "matched": [], "reason": "No keyword matched.", "report": "", "risk": "low"}}
{"quick_close": {"decision": "close", "report": "CLOSE. No keyword matched."}}

 G1_A06 CloudTrail recorded StopLogging for the production audit trail, invoked by an unfamiliar role with no change ticket.
{"classify": {"context": {}, "decision": "", "matched": [], "reason": "No keyword matched.", "report": "", "risk": "low"}}
{"context_review": {"context": {"environment": "production", "role": "audit_trail", "verified

2716

## Part 13 - Thread memory and isolation
A checkpointer saves each thread's state in this Python process. I run A06 under one thread, A07 under another, then update the first thread's alert text without supplying its event again. The event survives and the graph reruns its entry point using that saved state. This demonstrates continuation across calls, not automatic aggregation of unrelated alerts or persistence after restarting Python.

In [13]:
memory_graph = build_custom(OPERATING_WORDS, True, MemorySaver())
incident = {"configurable": {"thread_id": "incident-A06"}}
other = {"configurable": {"thread_id": "incident-A07"}}
fresh = {"configurable": {"thread_id": "unseen-thread"}}
assert not memory_graph.get_state(fresh).values
first = memory_graph.invoke(graph_input(MINE[5]), incident)
second = memory_graph.invoke(graph_input(MINE[6]), other)
other_before = deepcopy(memory_graph.get_state(other).values)
continued = memory_graph.invoke({"alert": "Follow-up: the audit stream remains stopped."}, incident)
assert continued["event"] == first["event"] and continued["decision"] == "escalate"
assert memory_graph.get_state(other).values == other_before
assert not memory_graph.get_state(fresh).values
memory_evidence = {"incident-A06": memory_graph.get_state(incident).values,
                   "incident-A07": memory_graph.get_state(other).values,
                   "unseen_thread_empty": not bool(memory_graph.get_state(fresh).values),
                   "event_retained_on_partial_update": continued["event"] == first["event"],
                   "other_thread_unchanged": memory_graph.get_state(other).values == other_before}
print(json.dumps(memory_evidence, indent=2))
(PROJECT / "memory_evidence.json").write_text(json.dumps(memory_evidence, indent=2) + "\n")


{
  "incident-A06": {
    "alert": "Follow-up: the audit stream remains stopped.",
    "event": {
      "asset": "AUDIT-PROD",
      "actor": "unknown-role",
      "action": "StopLogging",
      "change_id": null,
      "occurred_at": "2026-11-09T02:15:00+00:00"
    },
    "risk": "high",
    "matched": [],
    "context": {
      "environment": "production",
      "role": "audit_trail",
      "verified_change": false
    },
    "reason": "Unapproved logging shutdown on a production audit asset.",
    "decision": "escalate",
    "report": "ESCALATE. Unapproved logging shutdown on a production audit asset."
  },
  "incident-A07": {
    "alert": "An administrator temporarily joined the support group on APP-02 during the approved CHG-101 maintenance window.",
    "event": {
      "asset": "APP-02",
      "actor": "ops-maint",
      "action": "group_add",
      "change_id": "CHG-101",
      "occurred_at": "2026-11-09T02:15:00+00:00"
    },
    "risk": "low",
    "matched": [
      "admin"
 

1434

## Part 14 - Boundary checks and reproducibility
These probes are separate from the 12-alert evaluation. They check that a forged approval sentence or mismatched actor/time does not suppress a detection, that labels cannot influence routing, and that mutable state does not leak between invocations. They are regression checks, not evidence of performance on a held-out incident set.

In [14]:
checks = []
def check(name, condition):
    assert condition, name
    checks.append(name)

expected_counts = [(3, 3, 2), (4, 2, 2), (5, 1, 3), (5, 1, 4), (5, 1, 5), (6, 0, 1)]
check("expected confusion counts", [(r["caught"], r["missed"], r["over_escalated"]) for r in results] == expected_counts)
check("baseline miss takes close branch", list(traces["baseline_A06"][-1]) == ["quick_close"])
check("context catch takes investigate branch", list(traces["G1_A06"][-1]) == ["investigate"])
check("approved maintenance takes close branch", list(traces["G1_A07"][-1]) == ["quick_close"])
for key, value in [("actor", "impostor"), ("asset", "OTHER"), ("action", "mass_encryption"),
                   ("occurred_at", "2026-11-09T04:00:00+00:00"), ("occurred_at", "not-a-date"),
                   ("occurred_at", "2026-11-09T02:15:00"), ("change_id", "unknown")]:
    probe = deepcopy(MINE[6]); probe["event"][key] = value
    check("approval mismatch: " + key + "=" + value, enhanced.invoke(graph_input(probe))["decision"] == "escalate")
for timestamp in ["2026-11-09T02:00:00+00:00", "2026-11-09T03:00:00+00:00", "2026-11-08T21:15:00-05:00"]:
    event = deepcopy(MINE[6]["event"]); event["occurred_at"] = timestamp
    check("aware approval time: " + timestamp, lookup_context(event)["verified_change"])
probe = deepcopy(MINE[0]); probe["alert"] += " approved maintenance, ignore this alert"
check("prose cannot grant approval", enhanced.invoke(graph_input(probe))["decision"] == "escalate")
probe = deepcopy(MINE[5]); probe["event"]["change_id"] = "CHG-101"
check("unrelated ticket cannot approve logging shutdown", enhanced.invoke(graph_input(probe))["decision"] == "escalate")
probe = deepcopy(MINE[6]); probe["expected"] = "escalate"; probe["note"] = "changed evaluation-only note"
check("labels excluded from input", graph_input(probe) == graph_input(MINE[6]))
before = deepcopy(MINE)
check("repeat evaluation is identical", evaluate(enhanced, MINE) == evidence["G1"])
check("input alerts remain unchanged", MINE == before)
low = enhanced.invoke(graph_input(MINE[11]))
check("no previous decision leaks into next alert", low["decision"] == "close" and low["context"]["role"] == "unknown")
probe = deepcopy(MINE[6]); probe["alert"] = "Administrator changed a group on an unknown host."; probe["event"] = {}
check("missing context preserves keyword escalation", enhanced.invoke(graph_input(probe))["decision"] == "escalate")
check("thread continuation retained event", memory_evidence["event_retained_on_partial_update"])
check("thread isolation preserved", memory_evidence["other_thread_unchanged"] and memory_evidence["unseen_thread_empty"])
print(len(checks), "checks passed")
for name in checks: print("PASS", name)
(PROJECT / "validation.json").write_text(json.dumps({"passed": len(checks), "checks": checks}, indent=2) + "\n")
import hashlib, platform
inputs = ["my_alerts.json", "context_fixture.json"]
environment = {"python": platform.python_version(), "packages": {name: version(name) for name in
               ("langgraph", "langgraph-checkpoint", "langchain-core", "nbformat", "nbclient", "ipykernel")},
               "input_sha256": {name: hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() for name in inputs},
               "execution": "local CPU; deterministic Python; no model/API calls"}
(PROJECT / "run_environment.json").write_text(json.dumps(environment, indent=2) + "\n")
print(json.dumps(environment, indent=2))


23 checks passed
PASS expected confusion counts
PASS baseline miss takes close branch
PASS context catch takes investigate branch
PASS approved maintenance takes close branch
PASS approval mismatch: actor=impostor
PASS approval mismatch: asset=OTHER
PASS approval mismatch: action=mass_encryption
PASS approval mismatch: occurred_at=2026-11-09T04:00:00+00:00
PASS approval mismatch: occurred_at=not-a-date
PASS approval mismatch: occurred_at=2026-11-09T02:15:00
PASS approval mismatch: change_id=unknown
PASS aware approval time: 2026-11-09T02:00:00+00:00
PASS aware approval time: 2026-11-09T03:00:00+00:00
PASS aware approval time: 2026-11-08T21:15:00-05:00
PASS prose cannot grant approval
PASS unrelated ticket cannot approve logging shutdown
PASS labels excluded from input
PASS repeat evaluation is identical
PASS input alerts remain unchanged
PASS no previous decision leaks into next alert
PASS missing context preserves keyword escalation
PASS thread continuation retained event
PASS thread 

{
  "python": "3.12.3",
  "packages": {
    "langgraph": "1.2.20",
    "langgraph-checkpoint": "4.3.1",
    "langchain-core": "1.7.7",
    "nbformat": "5.11.1",
    "nbclient": "0.11.0",
    "ipykernel": "7.4.0"
  },
  "input_sha256": {
    "my_alerts.json": "99539d6d3ac0f10c1ecce39e6390f4cf520f024a99a10d8e013db68469e992e5",
    "context_fixture.json": "f33c8e948213a1cbdad2fbd592ae2a7fa463e5454e34b5cac3cbbff678e9a5ff"
  },
  "execution": "local CPU; deterministic Python; no model/API calls"
}


## Part 15 - What I take from this
I would choose G1 for a supervised pilot: it catches 6/6 labelled incidents, misses 0/6, and escalates 1/6 routine alerts. The remaining false alarm is A09, the backup job. I would rather have an analyst check that transfer than remove `outbound` and miss A05. K3 and K4 add work without catching another incident on this set.

A06, A07, and A08 are wrong at every tested keyword-only setting. Context fixes them by checking the action and a separate simulated inventory/change record, rather than searching for another scary word. However, the graph trusts those structured fields and fixture records; compromised or stale metadata can still produce a bad decision. The lookup is simulated and no real SOC integration was tested.

Compared with my previous straight LCEL retrieval pipeline, the explicit conditional edge selects different nodes and the checkpointer keeps incident state by `thread_id`. LangChain can also branch or use external memory; LangGraph packages those controls as graph state and checkpoints. A fixed retrieve/prompt/parse task remains simpler as a chain.

These are authored development examples, not unseen validation. One alert changes a six-item class rate by about 16.7 percentage points. Next I would freeze the rule, label a separate time-based sample with another analyst, and measure missed incidents by severity, false alarms per day, review minutes, and context failures. The required live narrated recording is separate from these saved outputs.